# Lab 6 (Free Edition): Building a Medical Chatbot with Gemini + ChromaDB

In this lab, you will build the **same pipeline as Lab 6** — but replacing the expensive AWS managed services with free alternatives:

| Lab 6 (Original) | Lab 6 (This Version) | What Changed |
|------------------|----------------------|--------------|
| Bedrock Knowledge Base | **ChromaDB** (local) | Free vector database |
| Amazon Titan Embeddings | **Gemini gemini-embedding-001** | Free embeddings |
| Claude via Bedrock | **Gemini 2.0 Flash** | Free LLM |
| Strands Agents | **Gemini function calling** | Built into the SDK |
| AgentCore Memory | **JSON file on disk** | Free persistence |
| AWS Lambda + S3 | ✅ **Same** | Still mostly free tier |
| LandingAI ADE | ✅ **Same** | Free tier |
| Visual Grounding | ✅ **Same** | PyMuPDF + Pillow |

**Learning Objectives:**
- Understand what a **vector database** is and how semantic search works
- Learn how **text embeddings** represent meaning as numbers
- Build an AI agent using **Gemini's function calling** API
- Implement **persistent memory** without any cloud service

## Outline

**Part 1: Setting Up the Lambda Function (Same as Original)**
- [Step 1: Environment Setup](#step1)
- [Step 2: Initialize AWS + Gemini Clients](#step2)
- [Steps 3–6: Deploy Lambda & S3 Trigger](#steps3-6) *(skip if already done)*
- [Step 7: Upload Documents & Monitor](#step7) *(skip if already done)*

**Part 2: Building the Free Knowledge Base**
- [Step 8: Initialize ChromaDB](#step8)
- [Step 9: Embed & Index Chunks with Gemini](#step9)

**Part 3: Building the Gemini Agent**
- [Step 10: Create the Search Tool with Visual Grounding](#step10)
- [Step 11: Set Up JSON Memory](#step11)
- [Step 12: Create the Gemini Agent](#step12)
- [Step 13: Interactive Chat](#step13)

## Installing Required Packages

We need a few extra packages compared to the original lab:
- **google-genai**: Google's official Python SDK for Gemini models
- **chromadb**: The open-source local vector database

We no longer need `bedrock-agentcore` or `strands-agents`.

In [ ]:
# Install all required packages
# google-genai  → Gemini API (embeddings + chat + function calling)
# chromadb      → local vector database (replaces Bedrock Knowledge Base)
# The rest are the same as Lab 6
!pip install --quiet boto3 python-dotenv Pillow PyMuPDF google-genai chromadb

<a id="step1"></a>

## Step 1: Environment Setup

We load credentials from a `.env` file. This file should live in the same folder as this notebook and must **never** be committed to Git (it's in `.gitignore`).

**Example `.env` file:**
```bash
# AWS (still needed for Lambda + S3)
AWS_ACCESS_KEY_ID=your_aws_access_key
AWS_SECRET_ACCESS_KEY=your_aws_secret_key
AWS_REGION=us-east-2
S3_BUCKET=your-bucket-name

# LandingAI ADE (get free key at https://bit.ly/3Ys8HXL)
VISION_AGENT_API_KEY=your_landingai_key

# Google Gemini (get free key at https://aistudio.google.com/)
GEMINI_API_KEY=your_gemini_api_key
```

> **Getting your Gemini API key**: Go to [Google AI Studio](https://aistudio.google.com/), sign in, and click "Get API key". The free tier gives you 15 requests/minute and 1,500 requests/day — more than enough for this lab.

In [ ]:
import boto3, os, json
from dotenv import load_dotenv

# Load environment variables from .env file
# After this call, os.getenv("GEMINI_API_KEY") etc. will work
_ = load_dotenv()

# Quick sanity check — print which vars are set (never print the values!)
required_vars = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION", "S3_BUCKET", "VISION_AGENT_API_KEY", "GEMINI_API_KEY"]
for var in required_vars:
    status = "✅ Set" if os.getenv(var) else "❌ MISSING"
    print(f"  {var}: {status}")

<a id="step2"></a>

## Step 2: Initialize AWS + Gemini Clients

We need two sets of clients:

**AWS clients** (same as original Lab 6):
| Client | Service | Purpose |
|--------|---------|--------|
| `s3_client` | Amazon S3 | Upload PDFs, download chunk JSONs |
| `lambda_client` | AWS Lambda | Deploy the document-parsing function |
| `iam` | IAM | Create Lambda permissions |
| `logs` | CloudWatch | Monitor Lambda execution |

**Gemini client** (new):
| Object | Purpose |
|--------|---------|
| `gemini_client` | Embeddings, chat, function calling |

**CONCEPT: Why a single `gemini_client` for everything?**  
The `google-genai` SDK uses one client object for all Gemini features:
- `client.models.embed_content(...)` → text embeddings
- `client.models.generate_content(...)` → text generation / chat

This simplifies the code compared to the original which needed separate `bedrock_runtime` and `bedrock_agent_runtime` clients.

In [ ]:
from google import genai
from google.genai import types

# ── AWS Clients (same as Lab 6) ──────────────────────────────────────────────
session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION"),
)

s3_client     = session.client("s3")
lambda_client = session.client("lambda")
iam           = session.client("iam")
logs          = session.client("logs")

# ── Gemini Client (new) ───────────────────────────────────────────────────────
# One client object handles both embedding and generation
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Quick test to confirm the Gemini API key works
try:
    test = gemini_client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=["hello world"]
    )
    print(f"✅ Gemini client ready (embedding dim: {len(test.embeddings[0].values)})")
except Exception as e:
    print(f"❌ Gemini client error: {e}")

print(f"✅ AWS clients ready (region: {os.getenv('AWS_REGION')})")

<a id="steps3-6"></a>

## Steps 3–6: Deploy Lambda & Set Up S3 Trigger

⏩ **These steps are identical to the original Lab 6.** If you already completed Lab 6 and your Lambda function (`ade-s3-handler`) is deployed go straight to Step 7.

If not, run the cells below. They:
1. Package `ade_s3_handler.py` + `landingai-ade` into a zip file
2. Create an IAM role for the Lambda function
3. Deploy the Lambda function to AWS
4. Configure S3 to trigger the Lambda on file uploads

In [ ]:
import pandas as pd
from lambda_helpers import *
print("Helper functions loaded")

In [ ]:
# Step 3: Create the deployment package
# Bundles ade_s3_handler.py + pip dependencies into ade_lambda.zip
zip_path = create_deployment_package(
    source_files=["ade_s3_handler.py"],
    requirements=["landingai-ade", "typing-extensions"],
    output_zip="ade_lambda.zip",
    package_dir="ade_package"
)

In [ ]:
# Step 4: Create IAM role (grants Lambda permission to read/write S3)
role_arn = create_or_update_lambda_role(
    iam_client=iam,
    role_name="lambda-ade-exec-role",
    description="Execution role for LandingAI ADE Lambda"
)

In [ ]:
# Step 5: Deploy the Lambda function
# Timeout: 900s (15 min) — large PDFs can take a while to parse
# Memory:  1024 MB — ADE needs some RAM for PDF processing
env_vars = {
    "VISION_AGENT_API_KEY": os.getenv("VISION_AGENT_API_KEY"),
    "ADE_MODEL": "dpt-2-latest",
    "INPUT_FOLDER": "input/",
    "OUTPUT_FOLDER": "output/",
    "S3_BUCKET": os.getenv("S3_BUCKET"),
    "FORCE_REPROCESS": "false"
}

response = deploy_lambda_function(
    lambda_client=lambda_client,
    function_name="ade-s3-handler",
    zip_file="ade_lambda.zip",
    role_arn=role_arn,
    handler="ade_s3_handler.ade_handler",
    env_vars=env_vars,
    runtime="python3.10",
    timeout=900,
    memory_size=1024
)

In [ ]:
# Step 6: Configure S3 to trigger Lambda on any upload to input/
setup_s3_trigger(
    s3_client=s3_client,
    lambda_client=lambda_client,
    bucket=os.getenv("S3_BUCKET"),
    prefix="input/",
    function_name="ade-s3-handler",
    suffix=None  # Tip: set to ".pdf" to only trigger on PDFs
)

<a id="step7"></a>

## Step 7: Upload Documents & Monitor Processing

⏩ **Skip this if your document chunks are already in S3** (from a previous run).

Upload the 8 documents to S3. Uploading to `input/documents/` automatically triggers the Lambda, which:
1. Downloads each PDF
2. Sends it to LandingAI ADE API → receives structured chunks with bounding boxes
3. Writes 3 output types to `output/`:
   - `output/documents/*.md` — full markdown
   - `output/documents_grounding/*.json` — all chunks with bounding boxes
   - `output/chunks/*.json` — **one file per chunk** (what we'll index into ChromaDB)

In [ ]:
# Upload PDFs from the local documents/ folder to S3 input/documents/
# skip_existing=True means it won't re-upload files already in S3
local_folder = "documents/"

if os.path.exists(local_folder):
    count = upload_folder_to_s3(
        s3_client=s3_client,
        local_folder=local_folder,
        s3_prefix=f"input/{local_folder}",
        bucket=os.getenv("S3_BUCKET"),
        file_extensions=[".pdf", ".PDF"]
    )
    print(f"\n⏳ Lambda is now parsing {count} PDFs in the background...")
    print("   Run the next cell to monitor progress.")
else:
    print(f"❌ Folder not found: {local_folder}")

In [ ]:
import json

# Manually invoke Lambda for each PDF already in S3
# This simulates what S3 would have sent as an event
bucket = os.getenv("S3_BUCKET")

# List all PDFs in input/documents/
resp = s3_client.list_objects_v2(Bucket=bucket, Prefix="input/documents/")
pdf_keys = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith(".pdf")]

print(f"Found {len(pdf_keys)} PDFs to process manually:\n")

for key in pdf_keys:
    # Build the same event shape that S3 sends to Lambda
    fake_event = {
        "Records": [{
            "s3": {
                "bucket": {"name": bucket},
                "object": {"key": key}
            }
        }]
    }
    
    print(f"⚡ Invoking Lambda for: {key.split('/')[-1]}")
    response = lambda_client.invoke(
        FunctionName="ade-s3-handler",
        InvocationType="Event",       # "Event" = async (fire and forget)
        Payload=json.dumps(fake_event)
    )
    status = response["StatusCode"]
    print(f"   Status: {status} {'✅' if status == 202 else '❌'}")

print(f"\n⏳ Lambda is now processing {len(pdf_keys)} PDFs in the background...")
print("Wait 2-5 minutes, then run the monitoring cell again.")


In [ ]:
# Watch Lambda processing in real-time via CloudWatch logs
# Press Esc then double-click 'i' to stop monitoring
stats = monitor_lambda_processing(
    logs_client=logs,
    s3_client=s3_client,
    bucket_name=os.getenv("S3_BUCKET")
)

<a id="step8"></a>

## Step 8: Initialize ChromaDB

### What is ChromaDB?

ChromaDB is an **open-source vector database** — the free alternative to Bedrock Knowledge Base.

A vector database stores text as **numerical vectors** (lists of floating-point numbers). When you search, it finds the vectors most *mathematically similar* to your query — this is how **semantic search** works.

```
Bedrock Knowledge Base          ChromaDB (this lab)
─────────────────────────       ─────────────────────────
Managed by AWS                  Runs locally on your machine
OpenSearch Serverless backend   HNSWlib (in-process, fast)
~$700/month minimum             FREE
Auto-scales                     Limited to your disk/RAM
```

### Persistence

We use `PersistentClient` which saves all data to the `./chroma_db/` folder on disk. The next time you open this notebook, all your indexed chunks are still there — no need to re-run the expensive embedding step.

### Cosine Similarity

We configure the collection to use **cosine distance** for comparisons. This measures the angle between two vectors regardless of their length — the standard for text similarity tasks.

```
cosine distance = 0   → vectors point in exactly the same direction (identical meaning)
cosine distance = 1   → vectors are perpendicular (unrelated)
cosine distance = 2   → vectors point in opposite directions (opposite meaning)
```

In [ ]:
from gemini_helpers import (
    init_chroma_collection,
    load_chunks_from_s3,
    embed_and_index_chunks,
    search_chroma,
    load_memory,
    save_memory,
    format_memory_for_prompt,
    update_memory_from_conversation
)

# Initialize the ChromaDB collection
# This creates (or reopens) a collection at ./chroma_db/
collection = init_chroma_collection(
    persist_directory="./chroma_db",  # Saved to disk in this folder
    collection_name="document_chunks"  # Like a table name
)

print(f"\nCollection info:")
print(f"  Name:      {collection.name}")
print(f"  Documents: {collection.count()}")

<a id="step9"></a>

## Step 9: Embed & Index Chunks with Gemini

### What happens in this step?

This step is the equivalent of the **Bedrock Knowledge Base ingestion job** from Lab 6. We're doing the same thing manually, which helps you understand exactly what happens under the hood:

```
S3: output/chunks/  →  Download JSONs  →  Gemini Embeddings  →  ChromaDB
     (raw chunk files)          (load_chunks)      (embed_texts)         (store)
```

### Gemini `gemini-embedding-001`

This is Google's state-of-the-art embedding model. It converts text into a **3072-dimensional vector** — a list of 768 numbers that encode the semantic meaning of the text.

**Free tier limits:**
- 1,500 embedding API calls per day
- 8 documents ≈ ~200–400 chunks total — well within the limit

### Idempotency (skip-existing)

The `embed_and_index_chunks()` function checks which chunk IDs are already in ChromaDB before embedding. Re-running this cell is safe — it only processes new chunks.

In [ ]:
# Step 9a: Download all chunk JSON files from S3
# These were created by the Lambda function (ade_s3_handler.py)
# Each file = one document chunk with: text, bbox, page, chunk_type, source_document
chunks = load_chunks_from_s3(
    s3_client=s3_client,
    bucket=os.getenv("S3_BUCKET"),
    chunks_prefix="output/chunks/"
)

# Preview one chunk so you can see the data structure
if chunks:
    print("\nExample chunk structure:")
    example = {k: v for k, v in chunks[0].items() if k != 's3_key'}  # Hide s3_key for brevity
    print(json.dumps(example, indent=2))

In [ ]:
# Step 9b: Embed each chunk with Gemini and store in ChromaDB
# This may take 1-2 minutes for ~200-400 chunks
num_indexed = embed_and_index_chunks(
    chunks=chunks,
    collection=collection,
    gemini_client=gemini_client,
    skip_existing=True  # Safe to re-run: skips already-indexed chunks
)

print(f"\nFinal collection size: {collection.count()} documents")

In [ ]:
# Step 9c: Quick test to verify the index works
# This embeds your query and finds the most semantically similar chunks
print("Testing semantic search...")
test_results = search_chroma(
    query="common cold symptoms",
    collection=collection,
    gemini_client=gemini_client,
    n_results=3
)

for i, result in enumerate(test_results):
    print(f"\n--- Result {i+1} (score: {result['score']:.3f}) ---")
    print(f"  Source: {result['source_document']}")
    print(f"  Page:   {result['page']}")
    print(f"  Type:   {result['chunk_type']}")
    print(f"  Text:   {result['text'][:200]}...")

<a id="step10"></a>

## Step 10: Create the Search Tool with Visual Grounding

### What is a "tool" in the context of AI agents?

An AI agent "thinks" by generating text, but it can also **call functions** (tools) to interact with the world:
- Search a database
- Run a calculation
- Call an API

The agent decides *when* to call a tool and *what arguments* to pass. It gets the tool's result back and incorporates it into its response.

In the original Lab 6, tools were registered using the `@strands.tool` decorator. With the Gemini SDK, we define tools using `types.FunctionDeclaration` — a JSON schema describing the function's name, description, and parameters.

### Visual Grounding (unchanged)

The visual grounding logic (`extract_chunk_image`) is **identical** to Lab 6. After finding the most relevant chunk in ChromaDB, we:
1. Download the source PDF from S3
2. Render the specific page
3. Crop to the chunk's bounding box coordinates
4. Add a red highlight border
5. Upload the cropped image to S3 and return a presigned URL

In [ ]:
from visual_grounding_helper import extract_chunk_image

def search_knowledge_base(query: str) -> str:
    """
    Search the document knowledge base for relevant information.
    Returns text content with visual grounding (page numbers + image URLs).

    This function replaces both the Bedrock Knowledge Base query AND the
    Strands tool decorator from the original Lab 6.

    Flow:
      1. Embed the query with Gemini gemini-embedding-001
      2. Search ChromaDB for the 5 most similar chunks (by cosine similarity)
      3. For each result: dynamically crop the PDF region & get a presigned URL
      4. Return formatted results with source, page, type, image URL, content
    """
    bucket = os.getenv("S3_BUCKET")

    # 1. Search ChromaDB using Gemini embeddings
    results = search_chroma(
        query=query,
        collection=collection,
        gemini_client=gemini_client,
        n_results=5
    )

    if not results:
        return f"No documents found for query: '{query}'. The knowledge base may be empty."

    # 2. Format each result with visual grounding
    formatted_results = []
    seen_chunk_ids = set()  # Deduplicate results

    for result in results:
        chunk_id       = result["chunk_id"]
        source_doc     = result["source_document"]
        score          = result["score"]
        page           = result["page"]
        chunk_type     = result["chunk_type"]
        bbox           = result["bbox"]
        content        = result["text"]

        if chunk_id in seen_chunk_ids:
            continue
        seen_chunk_ids.add(chunk_id)

        # 3. Generate a cropped chunk image for visual grounding
        # This downloads the source PDF, crops the region, adds a red border,
        # uploads the image to S3, and returns a presigned URL (1hr expiry)
        cropped_image_url = None
        if source_doc and bucket:
            source_pdf_key = f"input/documents/{source_doc}.pdf"
            try:
                s3_client.head_object(Bucket=bucket, Key=source_pdf_key)
                cropped_image_url = extract_chunk_image(
                    s3_client=s3_client,
                    bucket=bucket,
                    source_pdf_key=source_pdf_key,
                    bbox=bbox,
                    page_num=page,
                    chunk_id=chunk_id,
                    source_document=source_doc,
                    highlight=True,
                    padding=10
                )
            except Exception:
                pass  # PDF not found in S3, visual grounding skipped

        # 4. Format result string
        if cropped_image_url:
            result_text = f"""**Source:** {source_doc} (Relevance: {score:.2f})
📄 **Chunk ID:** {chunk_id}
📍 **Page:** {page}
🏷️ **Chunk Type:** {chunk_type}
🔍 **Visual Reference:** {cropped_image_url}

**Content:**
{content}"""
        else:
            result_text = f"""**Source:** {source_doc} (Relevance: {score:.2f})
📍 **Page:** {page} | 🏷️ **Type:** {chunk_type}

**Content:**
{content}"""

        formatted_results.append(result_text)

    return "\n\n---\n\n".join(formatted_results[:5])


# ─── Test the search function ────────────────────────────────────────────────
print("Testing search tool...")
test_result = search_knowledge_base("What are the main symptoms of the common cold?")
print("\n" + test_result[:500] + "...")

print("\n✅ Search tool is working!")

<a id="step11"></a>

## Step 11: Set Up JSON Memory

### Why does an agent need memory?

By default, each conversation session starts fresh — the agent has no idea what you discussed last time. AWS Bedrock AgentCore Memory solved this with a managed cloud service.

Our free alternative: a **JSON file on disk** (`memory.json`) that stores:

| What's stored | Example |
|--------------|--------|
| **Session summaries** | "We discussed Vitamin C evidence for cold prevention" |
| **User preferences** | `{"response_style": "short and bullet-pointed"}` |
| **Facts** | `["user is an analyst", "user prefers concise answers"]` |

At the start of each session, we read the file and inject the stored context into the system prompt. At the end of each session, we ask Gemini to extract a summary and any new preferences/facts, then save them back to the file.

This mimics all three of Bedrock AgentCore's memory strategies:
- `summaryMemoryStrategy` → our `session_summaries`
- `userPreferenceMemoryStrategy` → our `preferences`
- `semanticMemoryStrategy` → our `facts`

In [ ]:
# Load memory from disk (creates an empty memory dict if memory.json doesn't exist yet)
memory = load_memory("memory.json")

print("📚 Current memory state:")
print(f"  Sessions remembered: {len(memory['session_summaries'])}")
print(f"  Preferences stored:  {len(memory['preferences'])}")
print(f"  Facts extracted:     {len(memory['facts'])}")

if memory["session_summaries"]:
    print(f"\n  Last session: {memory['session_summaries'][-1]}")
else:
    print("\n  (No previous sessions — first run!)")

<a id="step12"></a>

## Step 12: Create the Gemini Agent

### How Gemini Function Calling Works

Instead of the Strands `Agent` class (which wraps AWS Bedrock + tool execution), we use Gemini's native **function calling** feature:

```
User: "What helps with cold symptoms?"
       ↓
Gemini thinks: "I should call search_knowledge_base"
       ↓
Gemini returns: FunctionCall(name='search_knowledge_base', args={'query': 'cold symptom remedies'})
       ↓
Our code executes the Python function and gets results
       ↓
We send results back to Gemini as a FunctionResponse
       ↓
Gemini generates a final answer using the retrieved content
```

This loop is called the **agent loop** or **reasoning loop**. We implement it ourselves in Step 13, which means you can see exactly what's happening at every step.

### Tool Declaration

We describe our tool to Gemini using `types.FunctionDeclaration` — a JSON schema that tells the model:
- What the function is called
- What it does (description)
- What parameters it accepts

Gemini reads this schema and decides on its own whether and when to call the function.

In [ ]:
from google.genai import types

# ── Tool Declaration ──────────────────────────────────────────────────────────
# This JSON schema tells Gemini about our search function.
# Gemini uses the "description" field to decide when to call it.
search_tool_declaration = types.FunctionDeclaration(
    name="search_knowledge_base",
    description=(
        "Search the document knowledge base for relevant information about "
        "the common cold, its symptoms, causes, treatments, and prevention. "
        "Returns text content with page numbers and visual references (image URLs) "
        "showing the exact location in the source PDF."
    ),
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "query": types.Schema(
                type=types.Type.STRING,
                description="The search query — a natural language question or topic"
            )
        },
        required=["query"]
    )
)

# Wrap the declaration in a Tool object
search_tool = types.Tool(function_declarations=[search_tool_declaration])


# ── System Prompt ─────────────────────────────────────────────────────────────
# Inject any stored memory context into the system prompt
memory_context = format_memory_for_prompt(memory)

SYSTEM_PROMPT = f"""You are a expert document analysis assistant specializing in the common cold.
You have access to a repository of documents via the search_knowledge_base tool.

Your capabilities:
- Search and analyze documents from the knowledge base
- Provide visual grounding: show EXACT page numbers and image URLs from source PDFs
- Remember user preferences and conversation history
- Provide evidence-based, cited responses

IMPORTANT: When search results include visual grounding information, you MUST include:
- The page number where information was found
- The visual reference URL (so the user can see the highlighted text in context)

Always call search_knowledge_base before answering questions about the documents.
Always cite your sources.
{memory_context}"""

# ── Agent Configuration ───────────────────────────────────────────────────────
AGENT_MODEL = "models/gemini-2.5-flash"  # Best free-tier model: 15 RPM, 1500 RPD

# Generation config — controls the LLM output style
GENERATION_CONFIG = types.GenerateContentConfig(
    temperature=0.3,        # Low temp = more factual, less creative (good for document info)
    max_output_tokens=2048, # Max response length
    tools=[search_tool],    # Register our search function
    system_instruction=SYSTEM_PROMPT
)

# Conversation history — grows as the user chats
# This is what gives the agent "short-term memory" within a session
conversation_history = []

print(f"✅ Gemini agent ready!")
print(f"   Model:   {AGENT_MODEL}")
print(f"   Tools:   [search_knowledge_base]")
print(f"   Memory:  {len(memory['session_summaries'])} previous sessions loaded")

<a id="step13"></a>

## Step 13: Interactive Chat

### The Agent Loop

Below we implement the **agent loop** manually — this is the core of how any AI agent works:

```
while True:
    1. Get user input
    2. Send to Gemini with conversation history
    3. If Gemini calls a tool → execute it → send result back → go to step 2
    4. If Gemini returns text → print it → add to history → go to step 1
```

This loop can repeat multiple times per user message (Gemini may decide to call the tool more than once). The original Strands agent framework hid this loop from you — here you see exactly how it works.

### Tips for testing
- `"What are the symptoms of the common cold?"`
- `"Does Vitamin C help prevent colds?"`
- `"I prefer short bullet-point answers"` (tests memory)
- `"What did we talk about?"` (tests short-term memory within session)
- Type `exit` to end the session (memory is saved automatically)

In [ ]:
from datetime import datetime
from google.genai import types as genai_types

# ── Tool Execution Map ────────────────────────────────────────────────────────
TOOL_MAP = {
    "search_knowledge_base": search_knowledge_base
}

def run_agent_turn(user_message: str) -> str:
    """
    Process one user message through the full agent loop.

    CONCEPT: Multi-turn function calling
    -------------------------------------
    A single user message may trigger multiple tool calls before Gemini
    produces a final text response. This while loop handles that case.

    WHY types.Content instead of plain dicts?
    ------------------------------------------
    The Gemini SDK v1.x uses pydantic v2 for strict type validation.
    Mixing plain dicts with SDK objects (types.Part) in the same list
    causes pydantic union validation to fail with cryptic "19 validation
    errors". Using types.Content throughout is explicit and always works.

    KEY SHORTCUT: After a model response, use candidate.content directly —
    it's already a types.Content object. No need to rebuild it from parts.

    Returns the final text response from Gemini.
    """
    # Add the user's message using types.Content (not a plain dict)
    conversation_history.append(
        genai_types.Content(role="user", parts=[genai_types.Part(text=user_message)])
    )

    # Agent loop: keep going until Gemini returns a text response
    while True:
        response = gemini_client.models.generate_content(
            model=AGENT_MODEL,
            contents=conversation_history,
            config=GENERATION_CONFIG
        )

        candidate = response.candidates[0]
        response_parts = candidate.content.parts

        # Check if Gemini wants to call a tool
        tool_calls = [p for p in response_parts if hasattr(p, 'function_call') and p.function_call]

        if tool_calls:
            # Reuse candidate.content directly — it's already a types.Content object
            conversation_history.append(candidate.content)

            # Execute each requested tool
            function_responses = []
            for part in tool_calls:
                fc = part.function_call
                tool_name = fc.name
                tool_args = dict(fc.args) if fc.args else {}

                print(f"   🔧 Calling tool: {tool_name}({', '.join(f'{k}={repr(v)}' for k,v in tool_args.items())})")

                if tool_name in TOOL_MAP:
                    tool_result = TOOL_MAP[tool_name](**tool_args)
                else:
                    tool_result = f"Error: Unknown tool '{tool_name}'"

                function_responses.append(
                    genai_types.Part.from_function_response(
                        name=tool_name,
                        response={"result": tool_result}
                    )
                )

            # Send tool results back — use types.Content, role="user"
            conversation_history.append(
                genai_types.Content(role="user", parts=function_responses)
            )

        else:
            # Gemini returned a text response — we're done
            final_text = "".join(
                p.text for p in response_parts
                if hasattr(p, 'text') and p.text
            )

            # Add Gemini's final response using types.Content
            conversation_history.append(
                genai_types.Content(role="model", parts=[genai_types.Part(text=final_text)])
            )

            return final_text


# ── Interactive Chat Loop ─────────────────────────────────────────────────────
# IMPORTANT: Reset conversation_history here so every run starts clean.
# Without this reset, re-running this cell accumulates old messages,
# causing "consecutive user messages" which pydantic rejects.
conversation_history = []

print("=" * 70)
print("Document Agent - Interactive Chat with Visual Grounding (Gemini)")
print("=" * 70)
print("\nAsk questions about documents.")
print("Type 'exit' to end the conversation (memory will be saved).")
print("=" * 70 + "\n")

conversation_num = 0

while True:
    try:
        user_input = input("\nYou: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ['exit', 'quit', 'bye', 'q']:
            print("\n⏳ Saving memory from this session...")
            memory = update_memory_from_conversation(
                memory=memory,
                conversation_history=conversation_history,
                gemini_client=gemini_client
            )
            save_memory(memory, "memory.json")
            print("\n👋 Goodbye! (Memory saved — I'll remember this next time)")
            break

        conversation_num += 1

        print("\n" + "─" * 70)
        print(f"Question #{conversation_num} [{datetime.now().strftime('%H:%M:%S')}]")
        print(f"  \"{user_input}\"")
        print("─" * 70)
        print("\nAgent Response:")
        print("  Processing...\n")

        result = run_agent_turn(user_input)
        print(result)
        print("\n" + "=" * 70)

    except KeyboardInterrupt:
        print("\n\n💾 Saving memory before exit...")
        memory = update_memory_from_conversation(memory, conversation_history, gemini_client)
        save_memory(memory, "memory.json")
        print("Conversation interrupted. Goodbye!")
        break
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("Please try again or type 'exit' to quit.")


## Summary

Here's what you built in this free version of Lab 6:

| Component | Service/Library | Cost | Function |
|-----------|----------------|------|----------|
| **Storage** | Amazon S3 | Free tier | Store raw PDFs and parsed outputs |
| **Parsing** | AWS Lambda + LandingAI ADE | Free tier / Free account | Serverless document chunking |
| **Vector DB** | ChromaDB (local) | Free | Semantic search over chunks |
| **Embeddings** | Gemini gemini-embedding-001 | Free (1500/day) | Text → 768-dim vectors |
| **Agent LLM** | Gemini 2.0 Flash | Free (1500/day) | Reasoning + function calling |
| **Memory** | JSON file on disk | Free | Cross-session persistence |
| **Visual Grounding** | PyMuPDF + Pillow | Free | Crop + annotate PDF regions |

**Key concepts you learned:**
- **Vector databases**: how semantic search works under the hood
- **Embeddings**: converting text meaning into numbers
- **The agent loop**: how function calling enables LLMs to use tools
- **Memory patterns**: storing and retrieving conversational context
- **Visual grounding**: linking AI answers back to source documents

**Extending this project:**
- Add more document types (invoices, contracts, reports)
- Add more tools (web search, calculator, external APIs)
- Use ChromaDB's filtering to search by document type or date range
- Deploy ChromaDB as a server for multi-user access